# 📊 K-Means Clustering — จัดกลุ่มจังหวัดตามโปรไฟล์คำขอ

**Goal:** จัดกลุ่ม 77 จังหวัดตามลักษณะการขอรับความช่วยเหลือ

**Features (8):**
- สัดส่วนคำขอแต่ละประเภท (4 features)
- อัตราอนุมัติ
- จำนวนคำขอรวม
- จำนวนเงินเฉลี่ยต่อคำขอ
- จำนวนเงินที่อนุมัติเฉลี่ย

In [ ]:
# Google Colab — run this cell first
!pip install pandas scikit-learn plotly openpyxl -q

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import plotly.graph_objects as go
import plotly.express as px

df = pd.read_excel('66-68Stat_ServiceType.xlsx')
df['RequestAmount'] = pd.to_numeric(df['RequestAmount'], errors='coerce').fillna(0)
df['TotalAmount'] = pd.to_numeric(df['TotalAmount'], errors='coerce').fillna(0)

CASE_SHORT = {
    'กรณีการขอปล่อยชั่วคราวผู้ต้องหาหรือจำเลย': 'ประกันตัว',
    'กรณีการช่วยเหลือประชาชนในการดำเนินคดี': 'ช่วยดำเนินคดี',
    'กรณีการช่วยเหลือผู้ถูกละเมิดสิทธิมนุษยชน หรือผู้ได้รับผลกระทบจากการถูกละเมิดสิทธิมนุษยชน': 'สิทธิมนุษยชน',
    'กรณีการสนับสนุนโครงการให้ความรู้ทางกฎหมายแก่ประชาชน': 'ความรู้กฎหมาย',
}
df['case_short'] = df['CaseTypeName'].map(CASE_SHORT)
print(f'Records: {len(df):,} | Provinces: {df["ProvinceName"].nunique()}')

## Feature Engineering — สร้าง Feature รายจังหวัด

In [ ]:
g = df.groupby('ProvinceName')
feat = pd.DataFrame(index=g.groups.keys())

# Total cases per province
tot = g['CaseAmount'].sum()

# Feature 1-4: proportion of each case type
for ct_full, ct_short in CASE_SHORT.items():
    sub = df[df['CaseTypeName']==ct_full].groupby('ProvinceName')['CaseAmount'].sum()
    feat[f'p_{ct_short}'] = (sub / tot).fillna(0)

# Feature 5: approval rate
approved = df[df['OpinionName']=='อนุมัติ'].groupby('ProvinceName')['CaseAmount'].sum()
feat['approval_rate'] = (approved / tot).fillna(0)

# Feature 6: total cases (log)
feat['log_cases'] = np.log1p(tot)

# Feature 7: average request amount
feat['avg_request'] = g['RequestAmount'].mean().fillna(0)

# Feature 8: average approved amount
feat['avg_approved'] = g['TotalAmount'].mean().fillna(0)

feat = feat.fillna(0)
print(f'Feature matrix: {feat.shape}')
feat.head()

## Elbow Method — เลือกจำนวนกลุ่ม

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(feat)

inertias = []
sil_scores = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X, km.labels_))

fig_elbow = go.Figure()
fig_elbow.add_trace(go.Scatter(x=list(K_range), y=inertias, mode='lines+markers',
    name='Inertia', line=dict(color='#264653', width=2)))
fig_elbow.add_trace(go.Scatter(x=list(K_range), y=sil_scores, mode='lines+markers',
    name='Silhouette', yaxis='y2', line=dict(color='#e76f51', width=2)))
fig_elbow.update_layout(
    title='Elbow Method + Silhouette Score',
    xaxis_title='k (จำนวนกลุ่ม)', yaxis_title='Inertia',
    yaxis2=dict(title='Silhouette', overlaying='y', side='right'),
    template='plotly_white', height=400,
    font=dict(family='Sarabun, sans-serif'),
)
fig_elbow.show()

## K-Means Clustering (k=4)

In [ ]:
K = 4
km = KMeans(n_clusters=K, random_state=42, n_init=10)
feat['cluster'] = km.fit_predict(X)

# PCA for visualization
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X)
pca_df = feat.copy()
pca_df['PC1'] = coords[:, 0]
pca_df['PC2'] = coords[:, 1]
pca_df['total'] = tot
pca_df['province'] = pca_df.index

sil = silhouette_score(X, feat['cluster'])
print(f'Silhouette Score: {sil:.4f}')
print(f'\nCluster sizes:')
print(feat['cluster'].value_counts().sort_index())

### Chart 1 — PCA Scatter: K-Means กลุ่มจังหวัด

ขนาดจุด = จำนวนคำขอ · สี = กลุ่ม

In [ ]:
CLUSTER_COLORS = ['#264653','#e76f51','#2a9d8f','#e9c46a','#606c38']

fig_pca = go.Figure()
for c in sorted(pca_df['cluster'].unique()):
    sub = pca_df[pca_df['cluster']==c]
    fig_pca.add_trace(go.Scatter(
        x=sub['PC1'], y=sub['PC2'], mode='markers+text',
        name=f'กลุ่ม {c}',
        marker=dict(size=sub['total'].apply(lambda x: max(12, min(50, np.log1p(x)*4))),
                    color=CLUSTER_COLORS[c % len(CLUSTER_COLORS)], opacity=.85,
                    line=dict(width=1, color='white')),
        text=sub['province'],
        textposition='top center', textfont=dict(size=8),
        hovertemplate='<b>%{text}</b><br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<extra>กลุ่ม '+str(c)+'</extra>',
    ))

fig_pca.update_layout(
    title=f'PCA 2D — K-Means จัดกลุ่ม {K} กลุ่ม (Silhouette: {sil:.3f})',
    xaxis_title=f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',
    yaxis_title=f'PC2 ({pca.explained_variance_ratio_[1]:.1%})',
    template='plotly_white', height=550,
    font=dict(family='Sarabun, sans-serif'),
)
fig_pca.show()

### Chart 2 — Cluster Profiles: แต่ละกลุ่มมีลักษณะอย่างไร?

In [ ]:
PALETTE = ['#264653','#2a9d8f','#e9c46a','#f4a261','#e76f51',
           '#606c38','#283618','#dda15e']
profile_cols = [c for c in feat.columns if c.startswith('p_') or c == 'approval_rate']
grp_means = feat.groupby('cluster')[profile_cols].mean()

fig_prof = go.Figure()
for i, col in enumerate(profile_cols):
    nm = col.replace('p_','')
    fig_prof.add_trace(go.Bar(
        name=nm, x=[f'กลุ่ม {c}' for c in grp_means.index],
        y=grp_means[col].values,
        marker_color=PALETTE[i % len(PALETTE)],
        hovertemplate=f'<b>{nm}</b><br>%{{y:.1%}}<extra></extra>',
    ))

fig_prof.update_layout(
    barmode='group',
    title='โปรไฟล์กลุ่ม — สัดส่วนประเภทคำขอ + อัตราอนุมัติ',
    yaxis_title='สัดส่วน', template='plotly_white', height=450,
    font=dict(family='Sarabun, sans-serif'),
    legend=dict(orientation='h', y=1.15),
)
fig_prof.show()

### Chart 3 — จังหวัดในแต่ละกลุ่ม

In [ ]:
for c in sorted(feat['cluster'].unique()):
    provs = feat[feat['cluster']==c].index.tolist()
    rate = feat[feat['cluster']==c]['approval_rate'].mean()
    print(f'\n📌 กลุ่ม {c} ({len(provs)} จังหวัด, อัตราอนุมัติเฉลี่ย: {rate:.1%})')
    print(f'   {" · ".join(provs[:10])}')
    if len(provs) > 10:
        print(f'   ... +{len(provs)-10} จังหวัด')

## สรุป

K-Means แบ่ง 77 จังหวัดเป็น 4 กลุ่มตามลักษณะการขอรับความช่วยเหลือจากกองทุนยุติธรรม

แต่ละกลุ่มมีลักษณะเฉพาะตัว เช่น กลุ่มที่มีสัดส่วนคำขอประกันตัวสูง vs กลุ่มที่เน้นช่วยเหลือดำเนินคดี